[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Inheritance


## What you will be able to do

Build a class on top of another, so that it inherits every method and changes only what differs,
call the parent's version of a method with `super()`, and read the order in which Python searches
classes for a method.


## The idea

### The problem

A program has a `Station` class that holds readings, works out a mean and writes a report. Then a
coastal station joins the network. It does everything a station does, and it also records the sea
temperature, which belongs in its report.

The quickest way to get a `CoastalStation` is to copy `Station`, rename it, and add the sea
readings. That works on the day it is written. From then on every method exists twice, and the two
copies drift apart: a bug fixed in `Station` is still in `CoastalStation`, because nobody remembered
the copy was there. The other quick way, a flag such as `is_coastal` inside `Station` and an `if` in
every method that differs, is the switchboard from the **Class and Static Methods** notebook again.

What is wanted is a way to say that a coastal station is a station with some differences, and to
write only the differences.

### What inheritance is

> **Inheritance** builds one class on another. Writing `class CoastalStation(Station):` makes
> `CoastalStation` a **subclass** of `Station`, its **parent**. A coastal station then has every
> method a station has, without a line being copied, and the subclass can add methods of its own or
> **override** one by defining a method with the same name. `super()` reaches the parent's version of
> a method, so an override can extend it rather than replace it.

### Why it works that way

When you call `coast.mean()`, Python looks for `mean` on the object, then on its class, then on
that class's parent, and so on up to `object`, the class every class ends at. That list of classes
is the **method resolution order**, and a class's `__mro__` attribute holds it. The first class that
has the method supplies it. So an inherited method is not a copy: there is one `mean`, on `Station`,
and every subclass finds it there. Fix it once, and every subclass is fixed.

An override works by being found first. `CoastalStation.report` comes earlier in the order than
`Station.report`, so it wins. `super().report()` continues the search from the next class along,
which is how an override reuses the original.

Because a coastal station is a station, `isinstance(coast, Station)` is `True`, and code written for
stations works on it. That is also where the promise from **Class and Static Methods** comes due: a
class method inherited by a subclass receives the subclass as `cls`, so `cls(...)` builds the right
kind of object, and `Station(...)` does not.

Inheritance is the right tool less often than it first appears, and the **Composition over
Inheritance** notebook makes that case. This one teaches it properly first.

### Where you will meet this

Every exception is a subclass. `FileNotFoundError` is an `OSError`, which is why `except OSError`
catches it, and the **Exceptions as Classes** notebook builds a family of your own. Every class you
have written inherits from `object`, which is where the default `__repr__` and `__eq__` in the
**Dunder Methods** notebook came from.

### What this notebook covers

A copied class against a subclass, put through the same numbered steps. Then what a subclass
inherits, how a method is looked up, extending `__init__` and overriding a method with `super()`,
`isinstance`, the `cls` promise, and multiple inheritance with its search order. Then one small
network of stations that uses all of it.

### A first look

A subclass with nothing in it. There is nothing to run yet: read it, and read the output underneath
it.

```python
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)


class CoastalStation(Station):
    pass


bergen = CoastalStation("Bergen", [3.1, 4.4])
print(bergen.mean())
print(isinstance(bergen, Station))
```

```
3.75
True
```

`CoastalStation` contains nothing of its own, and it can already do everything a `Station` can.


## Setup

Nothing to import this time. Every class in this notebook is written in the section that uses it.

**Run this cell before the rest of the notebook.**


In [1]:
print("ready")


ready


## Worked examples

### Before and after: a copied class, or a subclass

Here is the problem from the top of this notebook, in code. Both versions go through the same three
steps, numbered in the code and in the output:

1. Ask a station and a coastal station for their means.
2. Ask each one for its report.
3. After a bug is fixed, ask a new station and a new coastal station, neither with any readings yet,
   for their means. Both should say `None`.

First, the quick way: `CoastalStation` made by copying `Station` and adding the sea readings.


In [2]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"


class CoastalStation:
    def __init__(self, name, readings, sea):
        self.name = name
        self.readings = readings
        self.sea = sea

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        sea_mean = round(sum(self.sea) / len(self.sea), 2)
        return f"{self.name}: mean {self.mean()}, sea {sea_mean}"


Steps 1 and 2.


In [3]:
north = Station("Tromso", [-4.1, -2.6])
coast = CoastalStation("Bergen", [3.1, 4.4], sea=[7.2, 7.0])

# 1. Ask a station and a coastal station for their means.
print("1.", north.mean(), coast.mean())

# 2. Ask each one for its report.
print("2.", north.report())
print("  ", coast.report())


1. -3.35 3.75
2. Tromso: mean -3.35
   Bergen: mean 3.75, sea 7.1


Both work. Now a bug report: a new station with no readings yet crashes when asked for its mean,
because `mean` divides by `len(self.readings)`, which is zero. The fix goes where the bug was
reported, in `Station`: return `None` when there is nothing to average.


In [4]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def mean(self):
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"


Step 3.


In [5]:
# 3. After a bug is fixed, ask a new station and a new coastal station, neither with any
#    readings yet, for their means. Both should say None.
for new in [Station("Oslo", []), CoastalStation("Stavanger", [], sea=[])]:
    try:
        print("3.", new.name, new.mean())
    except ZeroDivisionError as error:
        print("3.", new.name, "crashed:", error)


3. Oslo None
3. Stavanger crashed: division by zero


The fix worked for Oslo and not for Stavanger. `CoastalStation` has its own copy of the old `mean`,
and redefining `Station` did nothing to it. Every copy of a method is another place a fix has to be
made, and nothing in the program reminds you that the copies exist.

Now the same two classes with inheritance. `Station` is the fixed version, and `CoastalStation`
contains only what is different about it.


In [6]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def mean(self):
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"


class CoastalStation(Station):
    def __init__(self, name, readings, sea):
        super().__init__(name, readings)
        self.sea = sea

    def report(self):
        sea_mean = round(sum(self.sea) / len(self.sea), 2)
        return f"{super().report()}, sea {sea_mean}"


The same three steps, character for character.


In [7]:
north = Station("Tromso", [-4.1, -2.6])
coast = CoastalStation("Bergen", [3.1, 4.4], sea=[7.2, 7.0])

# 1. Ask a station and a coastal station for their means.
print("1.", north.mean(), coast.mean())

# 2. Ask each one for its report.
print("2.", north.report())
print("  ", coast.report())

# 3. After a bug is fixed, ask a new station and a new coastal station, neither with any
#    readings yet, for their means. Both should say None.
for new in [Station("Oslo", []), CoastalStation("Stavanger", [], sea=[])]:
    try:
        print("3.", new.name, new.mean())
    except ZeroDivisionError as error:
        print("3.", new.name, "crashed:", error)


1. -3.35 3.75
2. Tromso: mean -3.35
   Bergen: mean 3.75, sea 7.1
3. Oslo None
3. Stavanger None


Step 3 is right for both stations this time, and the reason can be checked directly.


In [8]:
print("CoastalStation has a mean of its own:", "mean" in vars(CoastalStation))
print("coast.mean is Station's mean:        ", coast.mean.__func__ is Station.mean)


CoastalStation has a mean of its own: False
coast.mean is Station's mean:         True


`CoastalStation` has no `mean` at all. A coastal station uses `Station`'s, so there is exactly one
`mean` in the program, and exactly one place a fix to it can go.

| | Copied class | Subclass |
|---|---|---|
| Written for `CoastalStation` | `__init__`, `mean` and `report`, in full | the sea readings, and the sea part of the report |
| Copies of `mean` in the program | two | one |
| Step 3, after fixing `Station` | Stavanger crashed | Stavanger `None` |
| A later change to how every station works | made in every copy | made once, in `Station` |

The rest of this notebook takes the subclass version apart.

| Question about the subclass version | The section that answers it |
|---|---|
| What did `class CoastalStation(Station)` give it? | What a subclass inherits |
| How did `coast.mean()` find `Station`'s `mean`? | How a method is looked up |
| What does `super().__init__(name, readings)` do? | Extending `__init__` with `super()` |
| How does `report` add to `Station`'s report? | Overriding a method, and calling the original |
| Is a coastal station a station? | `isinstance` and `issubclass` |

### What a subclass inherits

A subclass with nothing in it at all is a complete class.


In [9]:
class AutomaticStation(Station):
    pass


auto = AutomaticStation("Svalbard", [-12.5, -14.0])

print("mean:  ", auto.mean())
print("report:", auto.report())
print("parent:", [c.__name__ for c in AutomaticStation.__bases__])
print("its own methods:", [name for name in vars(AutomaticStation) if not name.startswith("__")])


mean:   -13.25
report: Svalbard: mean -13.25
parent: ['Station']
its own methods: []


`AutomaticStation` has no methods of its own, and it still has `__init__`, `mean` and `report`, all
of them `Station`'s. `__bases__` lists the class or classes written in the parentheses.

### How a method is looked up

Python finds a method by searching a list of classes in order and taking the first one that has it.
`__mro__` holds the list. The loop below does the search by hand, the way Python does it.


In [10]:
print("CoastalStation.__mro__:", [c.__name__ for c in CoastalStation.__mro__])
print()

for name in ["report", "mean", "__init__", "__eq__"]:
    for owner in CoastalStation.__mro__:
        if name in vars(owner):
            print(f"{name:<9} is found on {owner.__name__}")
            break


CoastalStation.__mro__: ['CoastalStation', 'Station', 'object']

report    is found on CoastalStation
mean      is found on Station
__init__  is found on CoastalStation
__eq__    is found on object


`report` and `__init__` are found on `CoastalStation` itself, so its own versions win. `mean` is
not, so the search moves on to `Station`. `__eq__` is on neither, and is found on `object`, the class
every class ends at. The identity comparison that the **Dunder Methods** notebook met by default was
coming from here all along.

### Extending `__init__` with `super()`

`CoastalStation.__init__` has one extra job, storing `sea`, and the rest is `Station`'s job.
`super().__init__(name, readings)` runs `Station.__init__` on this same object, so the name and the
readings are stored exactly as `Station` stores them. Then the subclass adds its own attribute.


In [11]:
print(vars(coast))


{'name': 'Bergen', 'readings': [3.1, 4.4], 'sea': [7.2, 7.0]}


All three attributes are on one object. Two were stored by `Station`'s code and one by
`CoastalStation`'s, and none of `Station.__init__` had to be copied. Leaving the `super()` line out
is the first of the common errors at the end.

### Overriding a method, and calling the original

`CoastalStation.report` has the same name as `Station.report`, so it is found first and replaces it
for coastal stations. Inside it, `super().report()` runs `Station`'s version and returns its text,
and the override adds the sea to the end.


In [12]:
print("Station's report:       ", Station.report(coast))
print("CoastalStation's report:", coast.report())


Station's report:        Bergen: mean 3.75
CoastalStation's report: Bergen: mean 3.75, sea 7.1


The first line calls `Station.report` directly, passing the coastal station as `self`, which is the
call `super().report()` makes on the override's behalf. An override that did not call `super()`
would have to repeat the whole of `Station.report`, which is the copying this notebook set out to
avoid.

### `isinstance` and `issubclass`

A coastal station is a station, and Python can say so.


In [13]:
print("isinstance(coast, CoastalStation):  ", isinstance(coast, CoastalStation))
print("isinstance(coast, Station):         ", isinstance(coast, Station))
print("type(coast) is Station:             ", type(coast) is Station)
print("issubclass(CoastalStation, Station):", issubclass(CoastalStation, Station))
print("issubclass(Station, CoastalStation):", issubclass(Station, CoastalStation))


isinstance(coast, CoastalStation):   True
isinstance(coast, Station):          True
type(coast) is Station:              False
issubclass(CoastalStation, Station): True
issubclass(Station, CoastalStation): False


`isinstance` says yes, so anything that checks for a `Station` accepts a coastal one.
`type(coast) is Station` asks a stricter question, whether its class is exactly `Station`, and the
answer is no. Prefer `isinstance`, so that a subclass is welcome anywhere its parent is. The
relationship runs one way: a station is not a coastal station.

The same relationship runs through every exception you have caught.


In [14]:
print("FileNotFoundError.__mro__:", [c.__name__ for c in FileNotFoundError.__mro__])

try:
    open("no/such/readings.csv")
except OSError as error:
    print("except OSError caught a", type(error).__name__)


FileNotFoundError.__mro__: ['FileNotFoundError', 'OSError', 'Exception', 'BaseException', 'object']
except OSError caught a FileNotFoundError


`FileNotFoundError` is an `OSError`, which is an `Exception`. `except OSError` catches the whole
family, and the **Exceptions as Classes** notebook builds a family of your own.

### The promise from Class and Static Methods

The `from_csv` constructor in the **Class and Static Methods** notebook built its station with
`cls(...)`, and that notebook said the reason would come here. Here are two constructors side by
side, one using `cls` and one naming `Station`, inherited by a subclass that changes its report.


In [15]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

    def mean(self):
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])

    @classmethod
    def from_csv_named(cls, line):
        name, *values = line.split(",")
        return Station(name, [float(v) for v in values])


class AutomaticStation(Station):
    def report(self):
        return super().report() + " (automatic)"


built_with_cls = AutomaticStation.from_csv("Svalbard,-12.5,-14.0")
built_by_name = AutomaticStation.from_csv_named("Svalbard,-12.5,-14.0")

print("with cls(...):    ", built_with_cls, "|", built_with_cls.report())
print("with Station(...):", built_by_name, "|", built_by_name.report())


with cls(...):     AutomaticStation('Svalbard') | Svalbard: mean -13.25 (automatic)
with Station(...): Station('Svalbard') | Svalbard: mean -13.25


Both were called on `AutomaticStation`. The one that used `cls` built an `AutomaticStation`, because
`cls` was `AutomaticStation`, and its report says so. The one that named `Station` built a plain
station without any warning, and the subclass's report is gone.

`__repr__` makes the same choice for the same reason. It writes `type(self).__name__` rather than
the word `Station`, so each object prints under the name of its own class. A class that names itself
inside its own methods hands every subclass the wrong name.

### Multiple inheritance, and the order Python searches

A class can have more than one parent, as `class CoastalMountainSite(CoastalSite, MountainSite)`
does below. The method resolution order then decides which parent's method wins, and it also decides
what `super()` means, which is the part that surprises people.

Each class below adds its own word to a list and calls `super()` for the rest.


In [16]:
class Site:
    def describe(self):
        return ["site"]


class CoastalSite(Site):
    def describe(self):
        return ["coastal"] + super().describe()


class MountainSite(Site):
    def describe(self):
        return ["mountain"] + super().describe()


class CoastalMountainSite(CoastalSite, MountainSite):
    pass


class MountainCoastalSite(MountainSite, CoastalSite):
    pass


print("CoastalSite:        ", CoastalSite().describe())
print("CoastalMountainSite:", CoastalMountainSite().describe())
print("MountainCoastalSite:", MountainCoastalSite().describe())
print()
print("CoastalMountainSite.__mro__:", [c.__name__ for c in CoastalMountainSite.__mro__])


CoastalSite:         ['coastal', 'site']
CoastalMountainSite: ['coastal', 'mountain', 'site']
MountainCoastalSite: ['mountain', 'coastal', 'site']

CoastalMountainSite.__mro__: ['CoastalMountainSite', 'CoastalSite', 'MountainSite', 'Site', 'object']


Read the first two lines together. For a plain `CoastalSite`, `super()` inside
`CoastalSite.describe` reached `Site`. For a `CoastalMountainSite`, the same line of code reached
`MountainSite`, because `MountainSite` comes next in that object's search order.

So `super()` does not mean the parent. It means the next class in the order of the object's own
class, and that is how every class in the chain gets exactly one turn, with `Site` last and only
once. The parents are searched in the order they are written, which is why swapping them in
`MountainCoastalSite` put `mountain` first.

Multiple inheritance is used mostly for small add-on classes. Deep chains of it are hard to follow,
which is part of the argument in **Composition over Inheritance**.

### Putting it together: a small network

One base class and two subclasses, used together. `Station` supplies what every station shares: a
safe mean, a report, a constructor that uses `cls`, and a `__repr__` that uses `type(self)`.
`CoastalStation` extends `__init__` and its report, and `AutomaticStation` changes only its report.
The network is one list, and the loop that reports on it does not care which kind each station is.


In [17]:
class Station:
    """A weather station. The other classes here are built on it."""

    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

    def mean(self):
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])


class CoastalStation(Station):
    def __init__(self, name, readings, sea):
        super().__init__(name, readings)
        self.sea = sea

    def sea_mean(self):
        return round(sum(self.sea) / len(self.sea), 2)

    def report(self):
        return f"{super().report()}, sea {self.sea_mean()}"


class AutomaticStation(Station):
    def report(self):
        return super().report() + " (automatic, unattended)"


network = [
    Station("Tromso", [-4.1, -2.6]),
    CoastalStation("Bergen", [3.1, 4.4], sea=[7.2, 7.0]),
    AutomaticStation.from_csv("Svalbard,-12.5,-14.0"),
    Station("Oslo", []),
]

for station in network:
    print(f"{station!r:<28} {station.report()}")


Station('Tromso')            Tromso: mean -3.35
CoastalStation('Bergen')     Bergen: mean 3.75, sea 7.1
AutomaticStation('Svalbard') Svalbard: mean -13.25 (automatic, unattended)
Station('Oslo')              Oslo: mean None


One loop, three kinds of station, and each one reported in its own way, because each `report()` call
found the version its object's class supplies. The general name for this is **polymorphism**: code
written against the parent works on every subclass, without knowing which one it has. The empty Oslo
station reports `None` instead of crashing, because every station shares the one fixed `mean`.

`isinstance` picks out the coastal stations when only they will do.


In [18]:
print("coastal stations:", [s.name for s in network if isinstance(s, CoastalStation)])
print("all of them are stations:", all(isinstance(s, Station) for s in network))

with_readings = [s for s in network if s.mean() is not None]
print("coldest mean:", min(with_readings, key=lambda s: s.mean()).name)


coastal stations: ['Bergen']
all of them are stations: True
coldest mean: Svalbard


### Where each part came from

| In the network | What it relies on | The section that showed it |
|---|---|---|
| two subclasses without a copied line | a subclass inherits every method | What a subclass inherits |
| each `report()` finding its own class's version | the first class in the order supplies the method | How a method is looked up |
| `super().__init__(name, readings)` | the parent stores its own attributes | Extending `__init__` with `super()` |
| the coastal and automatic reports adding to the plain one | `super().report()` inside an override | Overriding a method, and calling the original |
| picking out the coastal stations | `isinstance` accepts subclasses | `isinstance` and `issubclass` |
| `AutomaticStation.from_csv` building an `AutomaticStation` | `cls(...)` rather than `Station(...)` | The promise from Class and Static Methods |
| each station printed under its own class name | `type(self).__name__` in `__repr__` | The promise from Class and Static Methods |
| Oslo reporting `None` | one shared `mean`, fixed once | Before and after |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/09-inheritance-solutions.ipynb).

**1.** Write `MountainStation(Station)`, whose `__init__` takes an extra `altitude`, stores the name
and readings through `super().__init__`, and stores `altitude` itself. Build one and print `vars` of
it.


In [19]:
# your code here


**2.** Override `report` in `MountainStation` so that it adds the altitude, reusing `Station`'s report
through `super()`.


In [20]:
# your code here


**3.** Print `isinstance(mountain, Station)` and `type(mountain) is Station`, and say in a comment
why the two answers differ.


In [21]:
# your code here


**4.** Print the class names in `MountainStation.__mro__`, then find which class supplies `mean` by
searching that list the way Python does.


In [22]:
# your code here


**5.** Build an `AutomaticStation` with the inherited `from_csv`, print its class name, and say in a
comment why it is not a plain `Station`.


In [23]:
# your code here


**6.** Put a `Station`, a `MountainStation` and an `AutomaticStation` in one list, and print every
report in a single loop.


In [24]:
# your code here


## Common errors

### AttributeError: the subclass `__init__` forgot `super().__init__`

Defining `__init__` in a subclass replaces the parent's entirely. If it does not call the parent's,
the parent's never runs.


In [25]:
class NoSuper(Station):
    def __init__(self, name, readings, sea):
        self.sea = sea


forgetful = NoSuper("Bergen", [3.1, 4.4], sea=[7.2, 7.0])
print("what it holds:", vars(forgetful))

forgetful.report()


what it holds: {'sea': [7.2, 7.0]}


AttributeError: 'NoSuper' object has no attribute 'name'

The object holds `sea` and nothing else. `Station.__init__` was never called, so the name and the
readings were never stored, and the error arrives at the first method that needs them rather than at
the line that caused it. A subclass `__init__` should call `super().__init__(...)` first, with the
arguments the parent needs.

### TypeError: `super().__init__` given `self` as well

`super()` already supplies the object, in the same way that `north.mean()` supplies `north`.


In [26]:
class TwiceSelf(Station):
    def __init__(self, name, readings, sea):
        super().__init__(self, name, readings)
        self.sea = sea


TwiceSelf("Bergen", [3.1, 4.4], sea=[7.2, 7.0])


TypeError: Station.__init__() takes 3 positional arguments but 4 were given

Passing `self` again gives `Station.__init__` one argument too many, and the count in the message is
one higher than you would expect, for the reason the **Your First Class** notebook gave: the object
was already on the list. The two spellings are easy to mix up. `Station.__init__(self, name, readings)`
does take `self`, because it is called on the class, and `super().__init__(name, readings)` does not.

### TypeError: an inherited constructor meets a changed `__init__`

`from_csv` builds with `cls(...)`, and `CoastalStation` inherits it. But `CoastalStation.__init__`
needs a `sea` argument that a CSV line of readings does not supply.


In [27]:
CoastalStation.from_csv("Bergen,3.1,4.4")


TypeError: CoastalStation.__init__() missing 1 required positional argument: 'sea'

`cls` built the right kind of object, and that kind needs an argument the constructor has no way to
provide. A subclass that changes what `__init__` takes has to override the constructors that call it
as well, or give the new argument a default. Every inherited method has to go on working for the
subclass, and changing `__init__` can break one without touching it.

### The quiet one: an override with a misspelled name

A method meant to override `report`, with its name typed wrongly.


In [28]:
class Typo(Station):
    def __init__(self, name, readings, sea):
        super().__init__(name, readings)
        self.sea = sea

    def reprot(self):
        sea_mean = round(sum(self.sea) / len(self.sea), 2)
        return f"{super().report()}, sea {sea_mean}"


bergen = Typo("Bergen", [3.1, 4.4], sea=[7.2, 7.0])

print("report():", bergen.report())
print("'report' in vars(Typo):", "report" in vars(Typo))
print("its own methods:", [name for name in vars(Typo) if not name.startswith("__")])


report(): Bergen: mean 3.75
'report' in vars(Typo): False
its own methods: ['reprot']


No error. `reprot` is a new method that nothing calls, and `report` is still found on `Station`, so
the coastal report quietly leaves out the sea. Nothing in the output looks wrong unless you already
know what the report should have said.

`vars` shows it at once: the subclass has a `reprot` and no `report`. Checking `"report" in
vars(Typo)` is the quickest way to confirm that an override actually took.


## Recap

- `class Child(Parent):` makes a subclass that has every method of its parent, without copying any.
- An inherited method is not a copy. There is one, on the parent, so a fix there fixes every
  subclass.
- Python looks a method up in the method resolution order, `__mro__`, and uses the first class that
  has it.
- Defining a method with the parent's name overrides it for the subclass.
- `super().method()` runs the next class's version, so an override can extend instead of replacing.
- A subclass `__init__` replaces the parent's, so call `super().__init__(...)` to store the parent's
  attributes.
- `super()` already supplies the object. Never pass it `self`.
- `isinstance(obj, Parent)` is `True` for subclasses. Prefer it to `type(obj) is Parent`.
- Every exception is part of a hierarchy, which is why `except OSError` catches `FileNotFoundError`.
- A class method inherited by a subclass receives the subclass as `cls`, so `cls(...)` builds the
  right class.
- `type(self).__name__` in `__repr__` prints each object under its own class's name.
- With several parents, `super()` means the next class in the search order, not the parent.
- A misspelled override raises nothing. Check with `"name" in vars(Class)`.


## What is next

The **Composition over Inheritance** notebook. Every subclass here was a genuine kind of station, and
inheritance fitted. Many hierarchies that get built are not like that: a class inherits from another
to borrow one method, and ends up tied to everything else the parent does. That notebook shows the
alternative, an object that holds another object instead of being one, and how to tell which you
need.


---

&#8592; **Previous:** [Class and Static Methods](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/08-class-and-static-methods.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Composition over Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/10-composition-over-inheritance.ipynb) &#8594;
